In [26]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [27]:
load_dotenv()

True

In [28]:
model=ChatOpenAI(model='gpt-4o-mini')

In [ ]:
class EvaluationScheme(BaseModel):
    feedback : str  =Field(description='detail feedback for the essay')
    score : int  =Field(description='Score out of 10',ge=0 , le=10)

    

In [30]:
structured_model=model.with_structured_output(EvaluationScheme)

In [ ]:
class EssayState(TypedDict):
    essay:str
    cot_textFeedback:str
    doa_textFeedback:str
    language_textFeedback:str
    individual_score:Annotated[list[int],operator.add]
    final_textFeedback:str
    avg_score:float

In [ ]:
def cot(state:EssayState)->EssayState:
   prompt = f"Provide feedback on the following essay using clarity of thought approach: {state['essay']}"
   response=structured_model.invoke(prompt)
   return {'cot_textFeedback':response.feedback,'individual_score':[response.score]}

def doa(state:EssayState)->EssayState:
    prompt = f"Provide feedback on the following essay using depth of analysis approach: {state['essay']}"
    response=structured_model.invoke(prompt)
    return {'doa_textFeedback':response.feedback,'individual_score':[response.score]}

def language(state:EssayState)->EssayState:
    prompt = f"Provide feedback on the following essay based on language: {state['essay']}"
    response=structured_model.invoke(prompt)
    return {'language_textFeedback':response.feedback,'individual_score':[response.score]}

def final(state:EssayState)->EssayState:
    prompt = f"Provide a overall feedback based on all these feeback {state['cot_textFeedback']},{state['doa_textFeedback']},{state['language_textFeedback']}"
    response=model.invoke(prompt).content
    avg=sum(state['individual_score'])/len(state['individual_score'])

    return {'final_textFeedback':response,'avg_score':avg}

In [33]:
graph =StateGraph(EssayState)

graph.add_node('cot',cot)
graph.add_node('doa',doa)
graph.add_node('language',language)
graph.add_node('final',final)


graph.add_edge(START,'cot')
graph.add_edge(START,'doa')
graph.add_edge(START,'language')
graph.add_edge('cot','final')
graph.add_edge('doa','final')
graph.add_edge('language','final')
graph.add_edge('final',END)

workflow = graph.compile()

In [34]:
essay="""Technology is very important in education because it helps students learn many things. Nowadays, technology is everywhere and students use it all the time. Technology has changed education in many ways and it is good for students because they can study online and use computers.

One reason technology is important is because it makes learning easier. Students can search anything on the internet and find information. This is useful because information is available. Also, technology helps teachers teach. Teachers can use presentations and videos, which makes classes better. Therefore, technology is important.

Another reason is that technology saves time. Students do not need to go to libraries because everything is online. This means they can finish their work faster. Faster work is better because it takes less time. Technology is also modern, and modern things are usually good.

Technology has some disadvantages too, but they are not very important. Some people say students get distracted by social media. This is true but technology is still helpful because students use it. Also, computers can sometimes stop working which causes problems.

In conclusion, technology is very important for education because it helps students learn, saves time, and is useful. Education and technology are connected together. Therefore, schools should use more technology because it is important and beneficial."""

In [ ]:
initial_state={'essay':essay }
final_state = workflow.invoke(initial_state)
print(final_state)